# 职业性价比排行

Cost-Performance Rankings — Best return on investment for career choices.

评分色阶：红色(低分0) → 黄色(中等5) → 绿色(高分10)

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Heiti TC', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

from pathlib import Path

csv_dir = Path('../data/csv')
all_files = sorted(csv_dir.glob('*.csv'))
print(f"Loading {len(all_files)} CSV files ...")

dfs = []
for f in all_files:
    tmp = pd.read_csv(f)
    dfs.append(tmp)
    print(f"  {f.name}: {len(tmp)} rows")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df)} rows, {df['sub_category'].nunique()} occupations, "
      f"{df['country_or_region'].nunique()} countries/regions")


In [ ]:
# Key columns for cost-performance analysis
base_cols = ['sub_category', 'sub_category_en', 'country_or_region', 'major_category', 'major_code', 'region']
cp_cols = ['cost_performance', 'learning_cost', 'education_req', 'value_added',
           'growth_coeff', 'opportunity', 'stability', 'composite_index']
display_cols = base_cols + cp_cols


## Global Top 50 Highest Cost-Performance

In [ ]:
top50_cp = df.nlargest(50, 'cost_performance')[display_cols].reset_index(drop=True)
top50_cp.index = top50_cp.index + 1
top50_cp.index.name = 'Rank'

score_gradient = [c for c in cp_cols if c in df.columns]
styled = top50_cp.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Best Cost-Performance by Education Level (typical_education)

In [ ]:
edu_levels = sorted(df['typical_education'].dropna().unique())
print(f"Education levels found: {len(edu_levels)}\n")

for edu in edu_levels:
    sub = df[df['typical_education'] == edu].nlargest(10, 'cost_performance')[display_cols].reset_index(drop=True)
    sub.index = sub.index + 1
    sub.index.name = 'Rank'
    print(f"\n{'='*60}")
    print(f"  {edu} (n={len(df[df['typical_education'] == edu])})")
    print(f"{'='*60}")
    styled = sub.style \
        .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
        .set_properties(**{'font-size': '11px'})
    display(styled)


## Best Cost-Performance per Country (Top 5 per Country)

In [ ]:
countries = sorted(df['country_or_region'].unique())
print(f"Countries/regions: {len(countries)}\n")

for country in countries:
    sub = df[df['country_or_region'] == country].nlargest(5, 'cost_performance')[display_cols].reset_index(drop=True)
    sub.index = sub.index + 1
    sub.index.name = 'Rank'
    print(f"\n{'='*60}")
    print(f"  {country}")
    print(f"{'='*60}")
    styled = sub.style \
        .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
        .set_properties(**{'font-size': '11px'})
    display(styled)


## Cost-Performance vs Learning Cost

Do low-cost-to-learn occupations have the best returns?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

categories = sorted(df['major_category'].unique())
colors = plt.cm.tab20(range(len(categories)))

for cat, color in zip(categories, colors):
    sub = df[df['major_category'] == cat]
    ax.scatter(sub['learning_cost'], sub['cost_performance'],
               alpha=0.4, s=15, label=cat, color=color)

ax.set_xlabel('Learning Cost (higher = more expensive)', fontsize=12)
ax.set_ylabel('Cost-Performance (higher = better value)', fontsize=12)
ax.set_title('Cost-Performance vs Learning Cost', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

corr = df[['cost_performance', 'learning_cost']].corr().iloc[0, 1]
print(f"\nCorrelation between Cost-Performance and Learning Cost: {corr:.3f}")
